# EXP-2026-001 runner: patient-conditioned MedSegFactory on RHUH-GBM

Thin runner. Every cell calls tested code in `rhuh_synthesis/` or a script in `scripts/`; no model, metric or split logic lives here.

- Run it from this repository on the machine that holds `data/RHUH-GBM`.
- Outputs stay under the Git-ignored `data/RHUH-GBM/synthesis/`. Nothing is uploaded.
- Cell outputs show patient IDs and MRI overlays. **Clear all outputs before saving or committing this notebook.**

Order: 1 environment, 2 status, 3 review flagged registrations and freeze the policy, 4 validation targets, 5 representation gate. Steps 3 and 5 need your decision. The test role stays sealed, and checkpoint loading and training are not run from here yet.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path.cwd().resolve()
if not (REPO / "rhuh_synthesis").is_dir():
    REPO = REPO.parent  # the notebook lives in notebooks/
assert (REPO / "rhuh_synthesis").is_dir(), "Open this notebook from the repository"
sys.path.insert(0, str(REPO))
PRIVATE = REPO / "data/RHUH-GBM/synthesis/EXP-2026-001"
CONFIG = json.loads((REPO / "configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
POLICY = PRIVATE / "registration_policy/acceptance_policy_v1.json"


def run(script, *args):
    """Run a repository script with this kernel's Python and show its output."""
    command = [sys.executable, str(REPO / "scripts" / script), *map(str, args)]
    print("$", " ".join(command[1:]))
    result = subprocess.run(command, cwd=REPO, capture_output=True, text=True)
    print(result.stdout[-6000:])
    print(result.stderr[-3000:])
    result.check_returncode()

## 1. Environment

In [ ]:
from rhuh_synthesis.provenance import code_version, config_sha256, environment

print(environment())
print(code_version(REPO), "| config sha256", config_sha256(CONFIG)[:12])
try:
    import torch

    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        capability = torch.cuda.get_device_capability(0)
        print(props.name, f"{props.total_memory / 2**30:.1f} GiB", "capability", capability)
        version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
        if capability >= (12, 0) and version < (2, 7):
            print("WARNING: a Blackwell GPU needs a CUDA 12.8 build of torch >= 2.7")
except ImportError:
    print("torch is not installed in this kernel")

## 2. Status (read-only)

In [ ]:
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
for role, info in split["roles"].items():
    print(f"{role}: {len(info['patients'])} patients, ID-3 positive {info['id3_positive']}, "
          f"pre-op scanner differs {info['scanner_differs']}")
qc = pd.concat(
    [pd.read_csv(path) for path in sorted((PRIVATE / "export_v1").glob("train_registration_qc*.csv"))],
    ignore_index=True,
)
columns = ["patient", "outline_dice_before", "outline_dice_after", "far_corr_before", "far_corr_after",
           "displacement_mean_mm", "id3_source_components", "id3_frame_components"]
qc[columns].round(3)

## 3. Review flagged registrations, then freeze the acceptance policy (decision 1)

Each overlay shows three axial levels. The green line is the pre-op brain outline.
- **Accept** when stable anatomy lines up after rigid registration: outline, ventricles, midline, far from the tumor. Differences next to the resection are expected postoperative change.
- **quarantine** removes the case from training.
- **affine_pending** also excludes it for now, until an affine variant is calibrated on training and frozen as a new policy version.

Validation/test cases are never reviewed one by one: under the frozen rule, a flagged held-out case is quarantined automatically.

In [ ]:
draft = json.loads((PRIVATE / "registration_policy/acceptance_policy_DRAFT.json").read_text(encoding="utf-8"))
print("flag rule:", draft["rule"])
flagged = draft["flagged_training_patients"]
overlay_dirs = sorted((PRIVATE / "export_v1/qc_overlays").glob("train*"))


def overlay(patient):
    return [d / f"{patient}.png" for d in overlay_dirs if (d / f"{patient}.png").exists()][-1]


typical = qc.sort_values("far_corr_after").iloc[len(qc) // 2]["patient"]
display(Markdown(f"**Typical unflagged pair for comparison: {typical}**"))
display(Image(filename=str(overlay(typical))))
for patient, values in flagged.items():
    display(Markdown(f"**{patient}**: far correlation {values['far_correlation_after']}, "
                     f"outline Dice {values['outline_dice_after']}"))
    display(Image(filename=str(overlay(patient))))

In [ ]:
# One decision per flagged patient: "accept", "quarantine" or "affine_pending".
DECISIONS = {patient: None for patient in flagged}
REVIEWER = ""    # your name or role
FREEZE = False   # True once every decision is filled; the policy file is written once

if FREEZE:
    from rhuh_synthesis.policy import freeze_acceptance_policy

    path, digest = freeze_acceptance_policy(
        PRIVATE / "registration_policy/acceptance_policy_DRAFT.json",
        DECISIONS, reviewer=REVIEWER, out_path=POLICY,
    )
    print("frozen:", path, digest[:12])
else:
    print("Not frozen yet. Fill DECISIONS and REVIEWER, then set FREEZE = True:", DECISIONS)

## 4. Register and export validation targets with the frozen policy

Runs only after step 3. Flagged validation cases are quarantined automatically and stay in the denominators. The test role is not exported here; it stays sealed until the locked analysis.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    run("register_and_export.py", "--role", "val", "--acceptance-policy", POLICY)

## 5. Representation gate on training references (decision 3)

This downloads the Stable Diffusion 1.5 VAE (about 335 MB, `stable-diffusion-v1-5/stable-diffusion-v1-5`) from Hugging Face. **Setting `ACCEPT_VAE_LICENSE = True` means you accept its CreativeML OpenRAIL-M license.**
- **Runtime:** minutes on a CUDA GPU; several hours on a CPU-only torch, as on this laptop now.
- **Exclusions:** quarantined and affine-pending training patients are taken from the frozen policy.
- **Pass rule:** every path (posterior mean and seeds 11/22/33) must pass on its own.

In [ ]:
ACCEPT_VAE_LICENSE = False
if ACCEPT_VAE_LICENSE:
    import torch

    excluded = json.loads(POLICY.read_text(encoding="utf-8"))["excluded_training"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    run("representation_gate.py", "--allow-download", "--device", device, "--exclude", *excluded)

In [ ]:
report_path = PRIVATE / "representation_gate_v1/report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("GATE", "PASS" if report["pass"] else "FAIL")
    display(pd.DataFrame(report["paths"]).T[
        ["macro_id3_dice", "macro_id3_recall", "mean_spurious_id3_ml_negative", "mean_unsupported_id1_ml", "pass"]
    ])
else:
    print("No representation-gate report yet.")

## Not run from this notebook yet

- **Stable-ROI coverage amendment (decision 2):** tell Claude/Codex which rule to record before any model scoring.
- **Checkpoint load, feature cache and training (Stages 3-5):** these need the GPU training host, approval to move data there, and an explicit training instruction.